### 📈 How to Calculate and Use Compound Annual Growth Rate (CAGR)

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 🧮 Geometric Returns

Regardless of the assets in our portfolio, we are subject to the geometric compounding of returns

 If you have a return series $r_1, r_2, ..., r_n$ (expressed as decimal returns, e.g. $r_1 = 0.07$ for 7%), the compounded (geometric) growth over $n$ periods is:
  $$
  \pi = P (1 + r_1) \cdot (1 + r_2) \cdot \ldots \cdot (1 + r_n) = P \times \prod_{i=1}^{n} (1 + r_i)
  $$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_PRINCIPAL = 1_000_000
NOMINAL_ANNUAL_RATE = 0.04
COMPOUNDS_PER_YEAR = 2
YEARS = 30  # went out to 30 years

# Make the animation smoother (lower frame duration)
FRAME_DURATION = 55  # was 180, lower = smoother
OUTPUT_HTML = "arithmetic_vs_geometric_30y_animation.html"

# ============================================================
# Build arithmetic and geometric schedules
# ============================================================

periodic_rate = NOMINAL_ANNUAL_RATE / COMPOUNDS_PER_YEAR
n_periods = YEARS * COMPOUNDS_PER_YEAR

# Semiannual curves for smooth animated lines
semi = pd.DataFrame({"period": np.arange(0, n_periods + 1)})
semi["year"] = semi["period"] / COMPOUNDS_PER_YEAR
semi["principal"] = INITIAL_PRINCIPAL
semi["arithmetic_total"] = INITIAL_PRINCIPAL * (1 + NOMINAL_ANNUAL_RATE * semi["year"])
semi["geometric_total"] = INITIAL_PRINCIPAL * (1 + periodic_rate) ** semi["period"]
semi["arithmetic_interest"] = semi["arithmetic_total"] - semi["principal"]
semi["geometric_interest"] = semi["geometric_total"] - semi["principal"]
semi["arithmetic_return"] = semi["arithmetic_total"] / INITIAL_PRINCIPAL - 1
semi["geometric_return"] = semi["geometric_total"] / INITIAL_PRINCIPAL - 1

# Annual bars for stacked principal + interest views
annual = pd.DataFrame({"year": np.arange(0, YEARS + 1)})
annual["period"] = annual["year"] * COMPOUNDS_PER_YEAR
annual["principal"] = INITIAL_PRINCIPAL
annual["arithmetic_total"] = INITIAL_PRINCIPAL * (1 + NOMINAL_ANNUAL_RATE * annual["year"])
annual["geometric_total"] = INITIAL_PRINCIPAL * (1 + periodic_rate) ** annual["period"]
annual["arithmetic_interest"] = annual["arithmetic_total"] - annual["principal"]
annual["geometric_interest"] = annual["geometric_total"] - annual["principal"]
annual["arithmetic_annual_interest"] = annual["arithmetic_total"].diff().fillna(0)
annual["geometric_annual_interest"] = annual["geometric_total"].diff().fillna(0)
annual["arithmetic_return"] = annual["arithmetic_total"] / INITIAL_PRINCIPAL - 1
annual["geometric_return"] = annual["geometric_total"] / INITIAL_PRINCIPAL - 1

# Summary metrics
arithmetic_final_value = annual["arithmetic_total"].iloc[-1]
geometric_final_value = annual["geometric_total"].iloc[-1]
arithmetic_final_interest = annual["arithmetic_interest"].iloc[-1]
geometric_final_interest = annual["geometric_interest"].iloc[-1]
geometric_excess = geometric_final_value - arithmetic_final_value
first_year_geometric_interest = annual.loc[annual["year"] == 1, "geometric_annual_interest"].iloc[0]
last_year_geometric_interest = annual.loc[annual["year"] == YEARS, "geometric_annual_interest"].iloc[0]
geometric_interest_acceleration = last_year_geometric_interest / first_year_geometric_interest

# ============================================================
# Styling — carried forward from the uploaded dark Plotly style
# ============================================================

off_white = "#e0e0e0"
principal_color = "#00ff88"
interest_color = "#ffaa33"
geometric_color = "#00d4ff"
baseline_color = "#777777"
arithmetic_color = "#f25b37"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_max = max(arithmetic_final_value, geometric_final_value) * 1.15

# ============================================================
# Subplot-specific metric strings
# ============================================================

left_subtitle = (
    "<b>Arithmetic Growth · Simple Interest</b><br>"
    f"Final Value: ${arithmetic_final_value:,.0f} · "
    f"Interest / Principal: {arithmetic_final_interest / INITIAL_PRINCIPAL:.2f}x"
)

right_subtitle = (
    "<b>Geometric Growth · Semiannual Compounding</b><br>"
    f"Final Value: ${geometric_final_value:,.0f} · "
    f"Year {YEARS} Interest vs Year 1: {geometric_interest_acceleration:.1f}x"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(left_subtitle, right_subtitle)
)

# -------------------------
# Left: Arithmetic growth
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[semi["year"].iloc[0]],
        y=[semi["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[semi["year"].iloc[0]],
        y=[semi["arithmetic_total"].iloc[0]],
        customdata=np.column_stack([
            semi["arithmetic_interest"].iloc[:1],
            semi["arithmetic_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=arithmetic_color, width=4),
        name="Arithmetic Total",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Arithmetic value: %{y:$,.0f}<br>"
            "Cumulative simple interest: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=[annual["year"].iloc[0]],
        y=[annual["principal"].iloc[0]],
        name="Principal",
        marker=dict(color=principal_color),
        opacity=0.42,
        showlegend=True,
        hovertemplate=(
            "Year: %{x}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=[annual["year"].iloc[0]],
        y=[annual["arithmetic_interest"].iloc[0]],
        customdata=np.column_stack([
            annual["arithmetic_annual_interest"].iloc[:1],
            annual["arithmetic_total"].iloc[:1],
            annual["arithmetic_return"].iloc[:1]
        ]),
        name="Simple Interest",
        marker=dict(color=interest_color),
        opacity=0.70,
        hovertemplate=(
            "Year: %{x}<br>"
            "Cumulative simple interest: %{y:$,.0f}<br>"
            "Interest earned that year: %{customdata[0]:$,.0f}<br>"
            "Total value: %{customdata[1]:$,.0f}<br>"
            "Cumulative return: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# -------------------------
# Right: Geometric growth
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[semi["year"].iloc[0]],
        y=[semi["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[semi["year"].iloc[0]],
        y=[semi["geometric_total"].iloc[0]],
        customdata=np.column_stack([
            semi["geometric_interest"].iloc[:1],
            semi["geometric_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=geometric_color, width=4),
        name="Geometric Total",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Geometric value: %{y:$,.0f}<br>"
            "Cumulative compounded interest: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual["year"].iloc[0]],
        y=[annual["principal"].iloc[0]],
        name="Principal",
        marker=dict(color=principal_color),
        opacity=0.42,
        showlegend=False,
        hovertemplate=(
            "Year: %{x}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual["year"].iloc[0]],
        y=[annual["geometric_interest"].iloc[0]],
        customdata=np.column_stack([
            annual["geometric_annual_interest"].iloc[:1],
            annual["geometric_total"].iloc[:1],
            annual["geometric_return"].iloc[:1]
        ]),
        name="Compounded Interest",
        marker=dict(color=interest_color),
        opacity=0.70,
        hovertemplate=(
            "Year: %{x}<br>"
            "Cumulative compounded interest: %{y:$,.0f}<br>"
            "Interest earned that year: %{customdata[0]:$,.0f}<br>"
            "Total value: %{customdata[1]:$,.0f}<br>"
            "Cumulative return: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Horizontal principal reference line on both charts
for col in [1, 2]:
    fig.add_hline(
        y=INITIAL_PRINCIPAL,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=col
    )

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

# One frame per year. Lines reveal semiannual points through that year.
for yr in range(0, YEARS + 1):
    frame_name = f"year_{yr}"

    semi_slice = semi[semi["year"] <= yr]
    annual_slice = annual[annual["year"] <= yr]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=semi_slice["year"],
                    y=semi_slice["principal"]
                ),
                go.Scatter(
                    x=semi_slice["year"],
                    y=semi_slice["arithmetic_total"],
                    customdata=np.column_stack([
                        semi_slice["arithmetic_interest"],
                        semi_slice["arithmetic_return"]
                    ])
                ),
                go.Bar(
                    x=annual_slice["year"],
                    y=annual_slice["principal"]
                ),
                go.Bar(
                    x=annual_slice["year"],
                    y=annual_slice["arithmetic_interest"],
                    customdata=np.column_stack([
                        annual_slice["arithmetic_annual_interest"],
                        annual_slice["arithmetic_total"],
                        annual_slice["arithmetic_return"]
                    ])
                ),
                go.Scatter(
                    x=semi_slice["year"],
                    y=semi_slice["principal"]
                ),
                go.Scatter(
                    x=semi_slice["year"],
                    y=semi_slice["geometric_total"],
                    customdata=np.column_stack([
                        semi_slice["geometric_interest"],
                        semi_slice["geometric_return"]
                    ])
                ),
                go.Bar(
                    x=annual_slice["year"],
                    y=annual_slice["principal"]
                ),
                go.Bar(
                    x=annual_slice["year"],
                    y=annual_slice["geometric_interest"],
                    customdata=np.column_stack([
                        annual_slice["geometric_annual_interest"],
                        annual_slice["geometric_total"],
                        annual_slice["geometric_return"]
                    ])
                )
            ],
            traces=[0, 1, 2, 3, 4, 5, 6, 7],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": str(yr),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

# Remove the main subtitle by removing the title
fig.update_layout(
    # No title or main subtitle
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=65, b=170, r=50, l=75),
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,           # Move below plot area
        xanchor="center",
        x=0.5,             # Centered horizontally
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": True},
                        "transition": {"duration": 80},  # Make play animation smoother
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 80},  # Smoother transition
        "pad": {"b": 40, "t": 30},  # Move the slider further down (was b:10, t:50)
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,   # Move slider downward by setting y to a negative value
        "steps": slider_steps
    }],
    annotations=list(fig.layout.annotations)
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, y_max],
    title_text="Principal + Cumulative Simple Interest",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, y_max],
    title_text="Principal + Cumulative Compounded Interest",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================
fig.show()


---

##### 🎲 Geometric Means Returns are Reinvested but the Rate is Random each Year 

Here's where things get tricky?  What do the following return profiles have in common.

 Two different return paths over three years, both starting from $1, can lead to the same (or similar) terminal value:

 $$
 \begin{align*}
 \pi_1 = & \quad 1 \times (1 + 0.50) \times (1 - 0.20) \times (1 + 0.10) = 1 \times 1.5 \times 0.8 \times 1.1 = 1.32 \\
 \pi_2 = & \quad 1 \times (1 + 0.10)^3 = 1 \times 1.1 \times 1.1 \times 1.1 = 1.33
 \end{align*}
 $$


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_PRINCIPAL = 1_000_000
NOMINAL_ANNUAL_RATE = 0.04
COMPOUNDS_PER_YEAR = 2
YEARS = 30

# Brownian bridge controls
# These paths are generated in LOG wealth space, then exponentiated.
# Because each bridge starts and ends at zero, both paths are forced to
# begin at INITIAL_PRINCIPAL and end at the same terminal value.
BRIDGE_VOLATILITY = 0.10
PATH_A_SEED = 95
PATH_B_SEED = 575

# Make the animation smoother (lower frame duration)
FRAME_DURATION = 55
OUTPUT_HTML = "brownian_bridge_same_cagr_animation.html"

# ============================================================
# Build Brownian bridge schedules
# ============================================================

periodic_rate = NOMINAL_ANNUAL_RATE / COMPOUNDS_PER_YEAR
shared_cagr = (1 + periodic_rate) ** COMPOUNDS_PER_YEAR - 1
terminal_value = INITIAL_PRINCIPAL * (1 + shared_cagr) ** YEARS
n_periods = YEARS * COMPOUNDS_PER_YEAR


def build_log_brownian_bridge_path(
    seed: int,
    volatility: float,
    label: str,
    initial_value: float = INITIAL_PRINCIPAL,
    final_value: float = terminal_value,
    years: int = YEARS,
    compounds_per_year: int = COMPOUNDS_PER_YEAR,
) -> pd.DataFrame:
    """
    Build a geometric wealth path using a Brownian bridge in log space.

    The log path is:
        log(V_t) = log(V_0) + target_log_return * (t / T) + sigma * bridge_t

    The bridge starts at 0 and ends at 0, so V_0 and V_T are fixed.
    That means every path generated here has the same geometric CAGR:
        CAGR = (V_T / V_0) ** (1 / T) - 1
    """

    n_steps = years * compounds_per_year
    dt = 1 / compounds_per_year
    time = np.arange(0, n_steps + 1) * dt

    rng = np.random.default_rng(seed)
    brownian_increments = rng.normal(loc=0, scale=np.sqrt(dt), size=n_steps)
    brownian_motion = np.concatenate([[0], np.cumsum(brownian_increments)])

    # Force the random process to finish at zero: B_T = 0
    bridge = brownian_motion - (time / years) * brownian_motion[-1]

    target_log_return = np.log(final_value / initial_value)
    log_total = (
        np.log(initial_value)
        + target_log_return * (time / years)
        + volatility * bridge
    )

    # Pin endpoints exactly to avoid tiny floating-point drift.
    log_total[0] = np.log(initial_value)
    log_total[-1] = np.log(final_value)

    total = np.exp(log_total)

    df = pd.DataFrame({
        "period": np.arange(0, n_steps + 1),
        "year": time,
        "principal": initial_value,
        "bridge": bridge,
        "total": total,
    })

    df["path"] = label
    df["cumulative_gain"] = df["total"] - df["principal"]
    df["cumulative_return"] = df["total"] / df["principal"] - 1
    df["log_cumulative_return"] = np.log(df["total"] / df["principal"])
    df["cagr_to_date"] = 0.0
    positive_time = df["year"] > 0
    df.loc[positive_time, "cagr_to_date"] = (
        (df.loc[positive_time, "total"] / df.loc[positive_time, "principal"])
        ** (1 / df.loc[positive_time, "year"])
        - 1
    )

    return df


path_a = build_log_brownian_bridge_path(
    seed=PATH_A_SEED,
    volatility=BRIDGE_VOLATILITY,
    label="Path A"
)

path_b = build_log_brownian_bridge_path(
    seed=PATH_B_SEED,
    volatility=BRIDGE_VOLATILITY,
    label="Path B"
)

# Same-CAGR deterministic reference curve shown on both charts
reference = pd.DataFrame({"period": np.arange(0, n_periods + 1)})
reference["year"] = reference["period"] / COMPOUNDS_PER_YEAR
reference["principal"] = INITIAL_PRINCIPAL
reference["total"] = INITIAL_PRINCIPAL * (1 + shared_cagr) ** reference["year"]
reference["cumulative_gain"] = reference["total"] - reference["principal"]
reference["cumulative_return"] = reference["total"] / reference["principal"] - 1

# Annual bars for stacked principal + cumulative path gain views
annual_a = path_a[path_a["period"] % COMPOUNDS_PER_YEAR == 0].copy()
annual_b = path_b[path_b["period"] % COMPOUNDS_PER_YEAR == 0].copy()

for annual_df in [annual_a, annual_b]:
    annual_df["annual_gain"] = annual_df["total"].diff().fillna(0)
    annual_df["annual_return"] = annual_df["total"].pct_change().fillna(0)
    annual_df["drawdown_from_peak"] = annual_df["total"] / annual_df["total"].cummax() - 1

# Summary metrics
path_a_final_value = annual_a["total"].iloc[-1]
path_b_final_value = annual_b["total"].iloc[-1]
path_a_cagr = (path_a_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
path_b_cagr = (path_b_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
path_a_max_value = path_a["total"].max()
path_b_max_value = path_b["total"].max()
path_a_min_value = path_a["total"].min()
path_b_min_value = path_b["total"].min()
path_a_max_drawdown = annual_a["drawdown_from_peak"].min()
path_b_max_drawdown = annual_b["drawdown_from_peak"].min()

# ============================================================
# Styling — carried forward from the uploaded dark Plotly style
# ============================================================

off_white = "#e0e0e0"
principal_color = "#00ff88"
gain_color = "#ffaa33"
path_a_color = "#f25b37"
path_b_color = "#00d4ff"
reference_color = "#aaaaaa"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_max = max(
    path_a["total"].max(),
    path_b["total"].max(),
    reference["total"].max(),
) * 1.15

# ============================================================
# Subplot-specific metric strings
# ============================================================

left_subtitle = (
    "<b>Brownian Bridge Path A</b><br>"
    f"Final Value: ${path_a_final_value:,.0f} · "
    f"CAGR: {path_a_cagr:.2%} · "
    f"Max DD: {path_a_max_drawdown:.1%}"
)

right_subtitle = (
    "<b>Brownian Bridge Path B</b><br>"
    f"Final Value: ${path_b_final_value:,.0f} · "
    f"CAGR: {path_b_cagr:.2%} · "
    f"Max DD: {path_b_max_drawdown:.1%}"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(left_subtitle, right_subtitle)
)

# -------------------------
# Left: Brownian bridge Path A
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[path_a["year"].iloc[0]],
        y=[path_a["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[reference["year"].iloc[0]],
        y=[reference["total"].iloc[0]],
        customdata=np.column_stack([
            reference["cumulative_gain"].iloc[:1],
            reference["cumulative_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dot"),
        opacity=0.85,
        name="Same CAGR Reference",
        showlegend=False,  # Hide Same CAGR Reference from legend (left)
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Same-CAGR value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[path_a["year"].iloc[0]],
        y=[path_a["total"].iloc[0]],
        customdata=np.column_stack([
            path_a["cumulative_gain"].iloc[:1],
            path_a["cumulative_return"].iloc[:1],
            path_a["cagr_to_date"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=path_a_color, width=4),
        name="Path A Wealth",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Path A value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "CAGR to date: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=[annual_a["year"].iloc[0]],
        y=[annual_a["principal"].iloc[0]],
        name="Principal",
        marker=dict(color=principal_color),
        opacity=0.42,
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=[annual_a["year"].iloc[0]],
        y=[annual_a["cumulative_gain"].iloc[0]],
        customdata=np.column_stack([
            annual_a["annual_gain"].iloc[:1],
            annual_a["annual_return"].iloc[:1],
            annual_a["total"].iloc[:1],
            annual_a["cumulative_return"].iloc[:1]
        ]),
        name="Path A Cumulative Gain",
        marker=dict(color=gain_color),
        opacity=0.70,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Cumulative gain: %{y:$,.0f}<br>"
            "Gain/loss that year: %{customdata[0]:$,.0f}<br>"
            "Annual return: %{customdata[1]:.2%}<br>"
            "Total value: %{customdata[2]:$,.0f}<br>"
            "Cumulative return: %{customdata[3]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# -------------------------
# Right: Brownian bridge Path B
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[path_b["year"].iloc[0]],
        y=[path_b["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[reference["year"].iloc[0]],
        y=[reference["total"].iloc[0]],
        customdata=np.column_stack([
            reference["cumulative_gain"].iloc[:1],
            reference["cumulative_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dot"),
        opacity=0.85,
        name="Same CAGR Reference",
        showlegend=False,  # Hide Same CAGR Reference from legend (right)
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Same-CAGR value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[path_b["year"].iloc[0]],
        y=[path_b["total"].iloc[0]],
        customdata=np.column_stack([
            path_b["cumulative_gain"].iloc[:1],
            path_b["cumulative_return"].iloc[:1],
            path_b["cagr_to_date"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=path_b_color, width=4),
        name="Path B Wealth",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Path B value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "CAGR to date: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual_b["year"].iloc[0]],
        y=[annual_b["principal"].iloc[0]],
        name="Principal",
        marker=dict(color=principal_color),
        opacity=0.42,
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual_b["year"].iloc[0]],
        y=[annual_b["cumulative_gain"].iloc[0]],
        customdata=np.column_stack([
            annual_b["annual_gain"].iloc[:1],
            annual_b["annual_return"].iloc[:1],
            annual_b["total"].iloc[:1],
            annual_b["cumulative_return"].iloc[:1]
        ]),
        name="Path B Cumulative Gain",
        marker=dict(color=gain_color),
        opacity=0.70,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Cumulative gain: %{y:$,.0f}<br>"
            "Gain/loss that year: %{customdata[0]:$,.0f}<br>"
            "Annual return: %{customdata[1]:.2%}<br>"
            "Total value: %{customdata[2]:$,.0f}<br>"
            "Cumulative return: %{customdata[3]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Horizontal principal reference line on both charts
for col in [1, 2]:
    fig.add_hline(
        y=INITIAL_PRINCIPAL,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=col
    )

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

# One frame per year. Lines reveal semiannual points through that year.
for yr in range(0, YEARS + 1):
    frame_name = f"year_{yr}"

    path_a_slice = path_a[path_a["year"] <= yr]
    path_b_slice = path_b[path_b["year"] <= yr]
    reference_slice = reference[reference["year"] <= yr]
    annual_a_slice = annual_a[annual_a["year"] <= yr]
    annual_b_slice = annual_b[annual_b["year"] <= yr]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=path_a_slice["year"],
                    y=path_a_slice["principal"]
                ),
                go.Scatter(
                    x=reference_slice["year"],
                    y=reference_slice["total"],
                    customdata=np.column_stack([
                        reference_slice["cumulative_gain"],
                        reference_slice["cumulative_return"]
                    ])
                ),
                go.Scatter(
                    x=path_a_slice["year"],
                    y=path_a_slice["total"],
                    customdata=np.column_stack([
                        path_a_slice["cumulative_gain"],
                        path_a_slice["cumulative_return"],
                        path_a_slice["cagr_to_date"]
                    ])
                ),
                go.Bar(
                    x=annual_a_slice["year"],
                    y=annual_a_slice["principal"]
                ),
                go.Bar(
                    x=annual_a_slice["year"],
                    y=annual_a_slice["cumulative_gain"],
                    customdata=np.column_stack([
                        annual_a_slice["annual_gain"],
                        annual_a_slice["annual_return"],
                        annual_a_slice["total"],
                        annual_a_slice["cumulative_return"]
                    ])
                ),
                go.Scatter(
                    x=path_b_slice["year"],
                    y=path_b_slice["principal"]
                ),
                go.Scatter(
                    x=reference_slice["year"],
                    y=reference_slice["total"],
                    customdata=np.column_stack([
                        reference_slice["cumulative_gain"],
                        reference_slice["cumulative_return"]
                    ])
                ),
                go.Scatter(
                    x=path_b_slice["year"],
                    y=path_b_slice["total"],
                    customdata=np.column_stack([
                        path_b_slice["cumulative_gain"],
                        path_b_slice["cumulative_return"],
                        path_b_slice["cagr_to_date"]
                    ])
                ),
                go.Bar(
                    x=annual_b_slice["year"],
                    y=annual_b_slice["principal"]
                ),
                go.Bar(
                    x=annual_b_slice["year"],
                    y=annual_b_slice["cumulative_gain"],
                    customdata=np.column_stack([
                        annual_b_slice["annual_gain"],
                        annual_b_slice["annual_return"],
                        annual_b_slice["total"],
                        annual_b_slice["cumulative_return"]
                    ])
                )
            ],
            traces=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": str(yr),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

# Remove the main subtitle by removing the title
fig.update_layout(
    # No title or main subtitle
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=65, b=170, r=50, l=75),
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": True},
                        "transition": {"duration": 80},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 80},
        "pad": {"b": 40, "t": 30},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }],
    annotations=list(fig.layout.annotations)
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, y_max],
    title_text="Principal + Cumulative Path Gain",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, y_max],
    title_text="Principal + Cumulative Path Gain",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================
fig.show()


If we aren't too concerned about the path (especially over long time horizons) how can we determine the effective rate we are compounding at every year?

Clearly, we aren't compounding at the realized rate of return each year.  Otherwise, if we could perform consistently in these capacities we would have exhorbanant wealth.

---

 ##### 📊 Compound Annual Growth Rate (CAGR)

 $$
 V_T = V_0 \cdot (1 + r)^T
 $$
 
 Solving for $r$:
 
 \begin{align*}
 \frac{V_T}{V_0} &= (1 + r)^T \\
 (1 + r) &= \left(\frac{V_T}{V_0}\right)^{1/T} \\
 r &= \left(\frac{V_T}{V_0}\right)^{1/T} - 1
 \end{align*}
 
 So, CAGR summarizes the total multi-year return as if growth each year had been steady at $r$.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_PRINCIPAL = 1_000_000
YEARS = 30
COMPOUNDS_PER_YEAR = 2

# Left panel: 100 Brownian bridges with the same terminal value / CAGR
BRIDGE_COUNT = 100
LEFT_TARGET_CAGR = 0.04
LEFT_BRIDGE_VOLATILITY = 0.18
LEFT_RANDOM_SEED = 2026

# Right panel: one S&P 500-style stochastic path with 7.5% terminal CAGR
SP500_TARGET_CAGR = 0.075
SP500_PATH_VOLATILITY = 0.16
SP500_PATH_SEED = 3

# Make the animation smoother (lower frame duration)
FRAME_DURATION = 55
OUTPUT_HTML = "brownian_bridge_fan_vs_sp500_animation.html"

# ============================================================
# Build Brownian bridge schedules
# ============================================================

n_periods = YEARS * COMPOUNDS_PER_YEAR
periods = np.arange(0, n_periods + 1)
time = periods / COMPOUNDS_PER_YEAR

def terminal_value_from_cagr(
    initial_value: float,
    cagr: float,
    years: int = YEARS,
) -> float:
    return initial_value * (1 + cagr) ** years


LEFT_TERMINAL_VALUE = terminal_value_from_cagr(INITIAL_PRINCIPAL, LEFT_TARGET_CAGR)
SP500_TERMINAL_VALUE = terminal_value_from_cagr(INITIAL_PRINCIPAL, SP500_TARGET_CAGR)


def build_log_brownian_bridge_path(
    seed: int,
    volatility: float,
    label: str,
    target_cagr: float,
    initial_value: float = INITIAL_PRINCIPAL,
    years: int = YEARS,
    compounds_per_year: int = COMPOUNDS_PER_YEAR,
) -> pd.DataFrame:
    """
    Build a geometric wealth path using a Brownian bridge in log space.

    The log path is:
        log(V_t) = log(V_0) + target_log_return * (t / T) + sigma * bridge_t

    Because the Brownian bridge starts at 0 and ends at 0, each simulated
    path is forced to start at V_0 and end at the target terminal value.
    Therefore the full-period geometric CAGR is exactly target_cagr, even
    though the interim return path is stochastic.
    """

    n_steps = years * compounds_per_year
    dt = 1 / compounds_per_year
    t = np.arange(0, n_steps + 1) * dt
    final_value = terminal_value_from_cagr(initial_value, target_cagr, years)

    rng = np.random.default_rng(seed)
    brownian_increments = rng.normal(loc=0, scale=np.sqrt(dt), size=n_steps)
    brownian_motion = np.concatenate([[0], np.cumsum(brownian_increments)])

    # Force the random process to finish at zero: bridge_0 = bridge_T = 0.
    bridge = brownian_motion - (t / years) * brownian_motion[-1]

    target_log_return = np.log(final_value / initial_value)
    log_total = (
        np.log(initial_value)
        + target_log_return * (t / years)
        + volatility * bridge
    )

    # Pin endpoints exactly to avoid tiny floating-point drift.
    log_total[0] = np.log(initial_value)
    log_total[-1] = np.log(final_value)

    total = np.exp(log_total)

    df = pd.DataFrame({
        "period": np.arange(0, n_steps + 1),
        "year": t,
        "principal": initial_value,
        "bridge": bridge,
        "total": total,
    })

    df["path"] = label
    df["cumulative_gain"] = df["total"] - df["principal"]
    df["cumulative_return"] = df["total"] / df["principal"] - 1
    df["log_cumulative_return"] = np.log(df["total"] / df["principal"])
    df["target_cagr"] = target_cagr
    df["cagr_to_date"] = 0.0

    positive_time = df["year"] > 0
    df.loc[positive_time, "cagr_to_date"] = (
        (df.loc[positive_time, "total"] / df.loc[positive_time, "principal"])
        ** (1 / df.loc[positive_time, "year"])
        - 1
    )

    return df


# 100 left-panel bridges. All have identical start/end values and the same CAGR.
left_bridge_paths = []
for i in range(BRIDGE_COUNT):
    left_bridge_paths.append(
        build_log_brownian_bridge_path(
            seed=LEFT_RANDOM_SEED + i,
            volatility=LEFT_BRIDGE_VOLATILITY,
            label=f"Bridge {i + 1:03d}",
            target_cagr=LEFT_TARGET_CAGR,
        )
    )

left_paths = pd.concat(left_bridge_paths, ignore_index=True)

# Right-panel S&P 500-style path. This is simulated, not historical data.
sp500_path = build_log_brownian_bridge_path(
    seed=SP500_PATH_SEED,
    volatility=SP500_PATH_VOLATILITY,
    label="S&P 500-Style Path",
    target_cagr=SP500_TARGET_CAGR,
)

# Deterministic CAGR reference curves
left_reference = pd.DataFrame({"period": periods, "year": time})
left_reference["principal"] = INITIAL_PRINCIPAL
left_reference["total"] = INITIAL_PRINCIPAL * (1 + LEFT_TARGET_CAGR) ** left_reference["year"]
left_reference["cumulative_gain"] = left_reference["total"] - left_reference["principal"]
left_reference["cumulative_return"] = left_reference["total"] / left_reference["principal"] - 1

sp500_reference = pd.DataFrame({"period": periods, "year": time})
sp500_reference["principal"] = INITIAL_PRINCIPAL
sp500_reference["total"] = INITIAL_PRINCIPAL * (1 + SP500_TARGET_CAGR) ** sp500_reference["year"]
sp500_reference["cumulative_gain"] = sp500_reference["total"] - sp500_reference["principal"]
sp500_reference["cumulative_return"] = sp500_reference["total"] / sp500_reference["principal"] - 1

# Annual bars for right-panel stacked principal + cumulative path gain view
annual_sp500 = sp500_path[sp500_path["period"] % COMPOUNDS_PER_YEAR == 0].copy()
annual_sp500["annual_gain"] = annual_sp500["total"].diff().fillna(0)
annual_sp500["annual_return"] = annual_sp500["total"].pct_change().fillna(0)
annual_sp500["drawdown_from_peak"] = annual_sp500["total"] / annual_sp500["total"].cummax() - 1

# Left-panel summary metrics
left_final_value = LEFT_TERMINAL_VALUE
left_final_cagr = (left_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
left_path_terminal_values = left_paths[left_paths["year"] == YEARS]["total"]
left_terminal_spread = left_path_terminal_values.max() - left_path_terminal_values.min()
left_min_value = left_paths["total"].min()
left_max_value = left_paths["total"].max()

# Right-panel summary metrics
sp500_final_value = annual_sp500["total"].iloc[-1]
sp500_cagr = (sp500_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
sp500_max_drawdown = annual_sp500["drawdown_from_peak"].min()
sp500_min_value = sp500_path["total"].min()
sp500_max_value = sp500_path["total"].max()

# ============================================================
# Styling — carried forward from the uploaded dark Plotly style
# ============================================================

off_white = "#e0e0e0"
principal_color = "#00ff88"
gain_color = "#ffaa33"
bridge_fan_color = "#ffaa33"
sp500_color = "#00d4ff"
reference_color = "#aaaaaa"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

all_visible_totals = pd.concat([
    left_paths["total"],
    sp500_path["total"],
    left_reference["total"],
    sp500_reference["total"],
])
y_max = all_visible_totals.max() * 1.12

# We'll cap the left chart's y-axis to $4,000,000 (4m).
LEFT_YMAX = 4_000_000

# ============================================================
# Subplot-specific metric strings
# ============================================================

left_subtitle = (
    "<b>100 Brownian Bridges · Same 4.00% CAGR Endpoint</b><br>"
    f"Final Value: ${left_final_value:,.0f} · "
    f"CAGR: {left_final_cagr:.2%} · "
    f"Terminal Spread: ${left_terminal_spread:,.0f}"
)

right_subtitle = (
    "<b>S&P 500-Style Stochastic Path · 7.50% CAGR</b><br>"
    f"Final Value: ${sp500_final_value:,.0f} · "
    f"CAGR: {sp500_cagr:.2%} · "
    f"Max DD: {sp500_max_drawdown:.1%}"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(left_subtitle, right_subtitle)
)

# -------------------------
# Left: 100 Brownian bridges, no bars
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[left_reference["year"].iloc[0]],
        y=[left_reference["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Remove 4% CAGR Reference from legend
fig.add_trace(
    go.Scatter(
        x=[left_reference["year"].iloc[0]],
        y=[left_reference["total"].iloc[0]],
        customdata=np.column_stack([
            left_reference["cumulative_gain"].iloc[:1],
            left_reference["cumulative_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dot"),
        opacity=0.85,
        name="4% CAGR Reference",
        showlegend=False,  # Remove from legend
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "4% CAGR value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Remove Brownian Bridges from legend
for i in range(BRIDGE_COUNT):
    bridge = left_bridge_paths[i]
    fig.add_trace(
        go.Scatter(
            x=[bridge["year"].iloc[0]],
            y=[bridge["total"].iloc[0]],
            text=bridge["path"].iloc[:1],
            customdata=np.column_stack([
                bridge["cumulative_gain"].iloc[:1],
                bridge["cumulative_return"].iloc[:1],
                bridge["cagr_to_date"].iloc[:1]
            ]),
            mode="lines",
            line=dict(color=bridge_fan_color, width=1.4),
            opacity=0.23,
            name="100 Brownian Bridges" if i == 0 else f"Bridge {i + 1:03d}",
            showlegend=False,  # Always hide all Brownian Bridges
            hovertemplate=(
                "%{text}<br>"
                "Year: %{x:.1f}<br>"
                "Value: %{y:$,.0f}<br>"
                "Cumulative gain: %{customdata[0]:$,.0f}<br>"
                "Cumulative return: %{customdata[1]:.2%}<br>"
                "CAGR to date: %{customdata[2]:.2%}<extra></extra>"
            )
        ),
        row=1,
        col=1
    )

# -------------------------
# Right: S&P 500-style stochastic path with bars
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[sp500_path["year"].iloc[0]],
        y=[sp500_path["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Remove 7.5% CAGR Reference from legend
fig.add_trace(
    go.Scatter(
        x=[sp500_reference["year"].iloc[0]],
        y=[sp500_reference["total"].iloc[0]],
        customdata=np.column_stack([
            sp500_reference["cumulative_gain"].iloc[:1],
            sp500_reference["cumulative_return"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dot"),
        opacity=0.85,
        name="7.5% CAGR Reference",
        showlegend=False,  # Remove from legend
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "7.5% CAGR value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[sp500_path["year"].iloc[0]],
        y=[sp500_path["total"].iloc[0]],
        customdata=np.column_stack([
            sp500_path["cumulative_gain"].iloc[:1],
            sp500_path["cumulative_return"].iloc[:1],
            sp500_path["cagr_to_date"].iloc[:1]
        ]),
        mode="lines",
        line=dict(color=sp500_color, width=4),
        name="S&P 500-Style Path",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "S&P 500-style value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "CAGR to date: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual_sp500["year"].iloc[0]],
        y=[annual_sp500["principal"].iloc[0]],
        name="Principal",
        marker=dict(color=principal_color),
        opacity=0.42,
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=[annual_sp500["year"].iloc[0]],
        y=[annual_sp500["cumulative_gain"].iloc[0]],
        customdata=np.column_stack([
            annual_sp500["annual_gain"].iloc[:1],
            annual_sp500["annual_return"].iloc[:1],
            annual_sp500["total"].iloc[:1],
            annual_sp500["cumulative_return"].iloc[:1]
        ]),
        name="S&P 500 Cumulative Gain",
        marker=dict(color=gain_color),
        opacity=0.70,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Cumulative gain: %{y:$,.0f}<br>"
            "Gain/loss that year: %{customdata[0]:$,.0f}<br>"
            "Annual return: %{customdata[1]:.2%}<br>"
            "Total value: %{customdata[2]:$,.0f}<br>"
            "Cumulative return: %{customdata[3]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Horizontal principal reference line on both charts
for col in [1, 2]:
    fig.add_hline(
        y=INITIAL_PRINCIPAL,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=col
    )

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

# Trace index map:
# 0: left principal baseline
# 1: left 4% CAGR reference
# 2..(BRIDGE_COUNT + 1): left bridge fan paths
# BRIDGE_COUNT + 2: right principal baseline
# BRIDGE_COUNT + 3: right 7.5% CAGR reference
# BRIDGE_COUNT + 4: right S&P 500-style path
# BRIDGE_COUNT + 5: right principal bars
# BRIDGE_COUNT + 6: right cumulative gain bars

for yr in range(0, YEARS + 1):
    frame_name = f"year_{yr}"

    left_reference_slice = left_reference[left_reference["year"] <= yr]
    sp500_reference_slice = sp500_reference[sp500_reference["year"] <= yr]
    sp500_path_slice = sp500_path[sp500_path["year"] <= yr]
    annual_sp500_slice = annual_sp500[annual_sp500["year"] <= yr]

    frame_data = [
        go.Scatter(
            x=left_reference_slice["year"],
            y=left_reference_slice["principal"]
        ),
        go.Scatter(
            x=left_reference_slice["year"],
            y=left_reference_slice["total"],
            customdata=np.column_stack([
                left_reference_slice["cumulative_gain"],
                left_reference_slice["cumulative_return"]
            ])
        ),
    ]

    for bridge in left_bridge_paths:
        bridge_slice = bridge[bridge["year"] <= yr]
        frame_data.append(
            go.Scatter(
                x=bridge_slice["year"],
                y=bridge_slice["total"],
                text=bridge_slice["path"],
                customdata=np.column_stack([
                    bridge_slice["cumulative_gain"],
                    bridge_slice["cumulative_return"],
                    bridge_slice["cagr_to_date"]
                ])
            )
        )

    frame_data.extend([
        go.Scatter(
            x=sp500_path_slice["year"],
            y=sp500_path_slice["principal"]
        ),
        go.Scatter(
            x=sp500_reference_slice["year"],
            y=sp500_reference_slice["total"],
            customdata=np.column_stack([
                sp500_reference_slice["cumulative_gain"],
                sp500_reference_slice["cumulative_return"]
            ])
        ),
        go.Scatter(
            x=sp500_path_slice["year"],
            y=sp500_path_slice["total"],
            customdata=np.column_stack([
                sp500_path_slice["cumulative_gain"],
                sp500_path_slice["cumulative_return"],
                sp500_path_slice["cagr_to_date"]
            ])
        ),
        go.Bar(
            x=annual_sp500_slice["year"],
            y=annual_sp500_slice["principal"]
        ),
        go.Bar(
            x=annual_sp500_slice["year"],
            y=annual_sp500_slice["cumulative_gain"],
            customdata=np.column_stack([
                annual_sp500_slice["annual_gain"],
                annual_sp500_slice["annual_return"],
                annual_sp500_slice["total"],
                annual_sp500_slice["cumulative_return"]
            ])
        ),
    ])

    trace_indices = list(range(BRIDGE_COUNT + 7))

    frames.append(
        go.Frame(
            data=frame_data,
            traces=trace_indices,
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": str(yr),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    # No title or main subtitle — matching the corrected style
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=65, b=170, r=50, l=75),
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": True},
                        "transition": {"duration": 80},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 80},
        "pad": {"b": 40, "t": 30},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }],
    annotations=list(fig.layout.annotations)
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, LEFT_YMAX],
    title_text="Wealth Path Value",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=1
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, y_max],
    title_text="Principal + Cumulative Path Gain",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.write_html(OUTPUT_HTML, include_plotlyjs="cdn", auto_play=False)
fig.show()


###### ______________________________________________________________________________________________________________________________________

##### 🃏 Tricks, Rules, and Reality

**Rule of 72:** The rule of 72 is particularly useful with CAGR.  Take 72 and divide it by CAGR, that's roughly how many years it takes for the principal to double at that rate.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_PRINCIPAL = 1_000_000
YEARS = 30

# Rule of 72 example: 7.5% CAGR
TARGET_CAGR = 0.075

# Monthly points make the curve smooth while the slider still advances annually.
POINTS_PER_YEAR = 12

# Make the animation smoother (lower frame duration)
FRAME_DURATION = 55
OUTPUT_HTML = "rule_of_72_75cagr_animation_fixed.html"

# ============================================================
# Build Rule of 72 schedule
# ============================================================

n_periods = YEARS * POINTS_PER_YEAR
periods = np.arange(0, n_periods + 1)
time = periods / POINTS_PER_YEAR

wealth = INITIAL_PRINCIPAL * (1 + TARGET_CAGR) ** time

curve = pd.DataFrame({
    "period": periods,
    "year": time,
    "principal": INITIAL_PRINCIPAL,
    "wealth": wealth,
})

curve["cumulative_gain"] = curve["wealth"] - curve["principal"]
curve["cumulative_return"] = curve["wealth"] / curve["principal"] - 1
curve["wealth_multiple"] = curve["wealth"] / curve["principal"]

curve["cagr_to_date"] = 0.0
positive_time = curve["year"] > 0
curve.loc[positive_time, "cagr_to_date"] = (
    (curve.loc[positive_time, "wealth"] / curve.loc[positive_time, "principal"])
    ** (1 / curve.loc[positive_time, "year"])
    - 1
)

# Rule of 72 approximation
rate_percent = TARGET_CAGR * 100
rule72_double_years = 72 / rate_percent
exact_double_years = np.log(2) / np.log(1 + TARGET_CAGR)

# Doubling ladder: 2x, 4x, 8x over 30 years at 7.5%
doubling_rows = []
for double_number, multiple in enumerate([2, 4, 8], start=1):
    exact_year = np.log(multiple) / np.log(1 + TARGET_CAGR)
    rule72_year = double_number * rule72_double_years
    value = INITIAL_PRINCIPAL * multiple

    if exact_year <= YEARS:
        doubling_rows.append({
            "double_number": double_number,
            "multiple": multiple,
            "label": f"{multiple}x",
            "value": value,
            "exact_year": exact_year,
            "rule72_year": rule72_year,
            "year_error": rule72_year - exact_year,
        })

doublings = pd.DataFrame(doubling_rows)

terminal_value = curve["wealth"].iloc[-1]
terminal_multiple = terminal_value / INITIAL_PRINCIPAL
terminal_return = terminal_value / INITIAL_PRINCIPAL - 1

# ============================================================
# Styling — carried forward from the uploaded dark Plotly style
# ============================================================

off_white = "#e0e0e0"
principal_color = "#00ff88"
wealth_color = "#00d4ff"
rule72_color = "#ffaa33"
exact_marker_color = "#f25b37"
reference_color = "#aaaaaa"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# NOTE: y_max is used before layout so we can place labels safely inside the chart.
# Top padding is large enough to keep the subtitle separate from all chart labels.
y_max = terminal_value * 1.16

# Interior label heights for the vertical Rule-of-72 guide labels.
# This avoids the previous overlap between the subtitle and the vline annotations.
rule72_label_y_positions = {
    2: y_max * 0.18,
    4: y_max * 0.27,
    8: y_max * 0.36,
}

# ============================================================
# Subplot metric string
# ============================================================

subtitle = (
    "<b>Rule of 72 · 7.50% CAGR Wealth Doubling</b><br>"
    f"72 ÷ 7.5 = {rule72_double_years:.1f} years per double · "
    f"Exact Double: {exact_double_years:.2f} years · "
    f"30Y Value: ${terminal_value:,.0f}"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=1,
    subplot_titles=(subtitle,)
)

# Principal baseline
fig.add_trace(
    go.Scatter(
        x=[curve["year"].iloc[0]],
        y=[curve["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# 7.5% CAGR wealth curve
fig.add_trace(
    go.Scatter(
        x=[curve["year"].iloc[0]],
        y=[curve["wealth"].iloc[0]],
        customdata=np.column_stack([
            curve["cumulative_gain"].iloc[:1],
            curve["cumulative_return"].iloc[:1],
            curve["wealth_multiple"].iloc[:1],
            curve["cagr_to_date"].iloc[:1],
        ]),
        mode="lines",
        line=dict(color=wealth_color, width=4),
        name="7.5% CAGR Wealth",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Wealth: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "Wealth multiple: %{customdata[2]:.2f}x<br>"
            "CAGR to date: %{customdata[3]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Animated current-year marker
fig.add_trace(
    go.Scatter(
        x=[curve["year"].iloc[0]],
        y=[curve["wealth"].iloc[0]],
        customdata=np.column_stack([
            curve["wealth_multiple"].iloc[:1],
            curve["cumulative_return"].iloc[:1],
        ]),
        mode="markers+text",
        marker=dict(color=wealth_color, size=10, line=dict(color=off_white, width=1)),
        text=[f"${curve['wealth'].iloc[0]:,.0f}"],
        textposition="top center",
        textfont=dict(color=off_white, size=12),
        name="Current Value",
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Current value: %{y:$,.0f}<br>"
            "Wealth multiple: %{customdata[0]:.2f}x<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Exact doubling points on the curve
fig.add_trace(
    go.Scatter(
        x=doublings["exact_year"],
        y=doublings["value"],
        customdata=np.column_stack([
            doublings["multiple"],
            doublings["rule72_year"],
            doublings["year_error"],
        ]),
        mode="markers",
        marker=dict(color=exact_marker_color, size=10, line=dict(color=off_white, width=1)),
        name="Exact Doubling Points",
        hovertemplate=(
            "%{customdata[0]:.0f}x Principal<br>"
            "Exact year: %{x:.2f}<br>"
            "Rule of 72 estimate: %{customdata[1]:.2f} years<br>"
            "Estimate error: %{customdata[2]:+.2f} years<br>"
            "Value: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Final endpoint marker
fig.add_trace(
    go.Scatter(
        x=[YEARS],
        y=[terminal_value],
        customdata=np.array([[terminal_multiple, terminal_return]]),
        mode="markers",
        marker=dict(color=rule72_color, size=12, symbol="diamond", line=dict(color=off_white, width=1)),
        name="30Y Endpoint",
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Final value: %{y:$,.0f}<br>"
            "Final multiple: %{customdata[0]:.2f}x<br>"
            "Total return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Horizontal multiple reference lines and Rule-of-72 vertical guide lines
for _, row in doublings.iterrows():
    multiple = int(row["multiple"])
    value = row["value"]
    exact_year = row["exact_year"]
    rule72_year = row["rule72_year"]

    # Only the multiple on the right, no value/million reference
    fig.add_hline(
        y=value,
        line=dict(color=reference_color, width=1, dash="dot"),
        opacity=0.55,
        annotation_text=f"{multiple}x",
        annotation_position="right",
        annotation_font=dict(color=off_white, size=11),
        row=1,
        col=1,
    )

    # Keep the vertical line itself, but do not attach labels at the top.
    fig.add_vline(
        x=rule72_year,
        line=dict(color=rule72_color, width=1.5, dash="dash"),
        opacity=0.75,
        row=1,
        col=1,
    )

    # Place Rule-of-72 labels inside the plot at staggered y-values.
    fig.add_annotation(
        x=rule72_year,
        y=rule72_label_y_positions[multiple],
        xref="x",
        yref="y",
        text=f"{multiple}x ≈ {rule72_year:.1f}Y",
        showarrow=False,
        align="center",
        bgcolor="rgba(30,30,30,0.82)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        borderpad=4,
        font=dict(color=rule72_color, size=11),
    )

# Horizontal principal reference line
fig.add_hline(
    y=INITIAL_PRINCIPAL,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

# One frame per year. The line reveals monthly points through that year.
for yr in range(0, YEARS + 1):
    frame_name = f"year_{yr}"

    curve_slice = curve[curve["year"] <= yr]
    current_row = curve[curve["year"] == yr].iloc[0]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=curve_slice["year"],
                    y=curve_slice["principal"],
                ),
                go.Scatter(
                    x=curve_slice["year"],
                    y=curve_slice["wealth"],
                    customdata=np.column_stack([
                        curve_slice["cumulative_gain"],
                        curve_slice["cumulative_return"],
                        curve_slice["wealth_multiple"],
                        curve_slice["cagr_to_date"],
                    ])
                ),
                go.Scatter(
                    x=[current_row["year"]],
                    y=[current_row["wealth"]],
                    customdata=np.array([[
                        current_row["wealth_multiple"],
                        current_row["cumulative_return"],
                    ]]),
                    text=[f"${current_row['wealth']:,.0f}<br>{current_row['wealth_multiple']:.2f}x"],
                ),
            ],
            traces=[0, 1, 2],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": str(yr),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    # No main title — matching the corrected style
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,

    # Extra top margin keeps the subtitle away from the plot body.
    # The Rule-of-72 labels are now inside the plot, not at the top edge.
    margin=dict(t=95, b=170, r=55, l=75),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": True},
                        "transition": {"duration": 80},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 80},
        "pad": {"b": 40, "t": 30},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }],
    annotations=list(fig.layout.annotations)
)

fig.update_annotations(font=dict(color=off_white, size=14))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=2
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, y_max],
    title_text="Wealth Value",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================
fig.show()

###### ______________________________________________________________________________________________________________________________________

**BS Test:** Finance can be like the gym.  It's hard to tell if results are real (*returns, hard work*) or fake (*overfit, steroids*)

There's no free lunch, if you are getting an outrageous return you are likely taking outrageous risk...

In [ ]:
principal = 100000
rate = 1  # 50% annual return
years = 30

final_value = principal * (1 + rate) ** years
print(f"Principal after 30 years at 100% CAGR: ${final_value:,.0f}")

###### ______________________________________________________________________________________________________________________________________

**Reality of CAGR:** Historical measures are not indicative of what's to come, it literally happened already, nobody knows what the future holds.  Historical proxies could perfectly predict what happens or have literally nothing to do with what happens over the long haul, these frameworks should be used to evaluate different states of the world, **NOT** predict anything.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

INITIAL_PRINCIPAL = 1_000_000
YEARS = 30
COMPOUNDS_PER_YEAR = 2

# Cited / forward-looking CAGR assumption
SPX_CAGR = 0.075

# Drawdown scenario
DRAWDOWN_YEAR = 5
UNHEDGED_DRAWDOWN = 0.40       # -40% market drawdown at year 5
HEDGED_DRAWDOWN = 0.15         # crisis alpha cuts drawdown to -15% at year 5

# Final year drawdown override (NEW)
FINAL_UNHEDGED_DRAWDOWN = 0.40  # -40% at year 30
FINAL_HEDGED_DRAWDOWN = 0.15    # -15% at year 30

# Stochastic path controls
# These paths are generated in LOG wealth space.
LEFT_SPX_VOLATILITY = 0.16
PRE_SHOCK_VOLATILITY = 0.14
UNHEDGED_RECOVERY_VOLATILITY = 0.20
HEDGED_RECOVERY_VOLATILITY = 0.14

LEFT_SPX_SEED = 17
PRE_SHOCK_SEED = 2026
UNHEDGED_RECOVERY_SEED = 404
HEDGED_RECOVERY_SEED = 1515

# Animation controls
FRAME_DURATION = 55
OUTPUT_HTML = "spx_stochastic_drawdown_crisis_alpha_animation.html"

# ============================================================
# Helpers
# ============================================================

def terminal_value_from_cagr(
    initial_value: float,
    cagr: float,
    years: int,
) -> float:
    return initial_value * (1 + cagr) ** years


def build_log_brownian_bridge_segment(
    start_year: float,
    end_year: float,
    start_value: float,
    end_value: float,
    volatility: float,
    seed: int,
    compounds_per_year: int = COMPOUNDS_PER_YEAR,
) -> pd.DataFrame:
    """
    Build a stochastic geometric wealth segment using a Brownian bridge
    in log wealth space.

    The path is pinned to start_value at start_year and end_value at end_year,
    while allowing stochastic variation between those endpoints.
    """

    start_period = int(round(start_year * compounds_per_year))
    end_period = int(round(end_year * compounds_per_year))
    segment_periods = np.arange(start_period, end_period + 1)
    segment_years = segment_periods / compounds_per_year

    n_steps = len(segment_periods) - 1
    if n_steps == 0:
        return pd.DataFrame({
            "period": segment_periods,
            "year": segment_years,
            "total": [start_value],
        })

    dt = 1 / compounds_per_year
    segment_length_years = end_year - start_year
    local_time = segment_years - start_year

    rng = np.random.default_rng(seed)
    brownian_increments = rng.normal(loc=0, scale=np.sqrt(dt), size=n_steps)
    brownian_motion = np.concatenate([[0], np.cumsum(brownian_increments)])

    # Brownian bridge: force random process to be zero at both endpoints.
    bridge = brownian_motion - (local_time / segment_length_years) * brownian_motion[-1]

    target_log_return = np.log(end_value / start_value)
    log_total = (
        np.log(start_value)
        + target_log_return * (local_time / segment_length_years)
        + volatility * bridge
    )

    # Pin endpoints exactly.
    log_total[0] = np.log(start_value)
    log_total[-1] = np.log(end_value)

    return pd.DataFrame({
        "period": segment_periods,
        "year": segment_years,
        "total": np.exp(log_total),
    })


def add_path_metrics(
    df: pd.DataFrame,
    label: str,
    principal: float = INITIAL_PRINCIPAL,
) -> pd.DataFrame:
    df = df.copy()
    df["principal"] = principal
    df["path"] = label
    df["cumulative_gain"] = df["total"] - df["principal"]
    df["cumulative_return"] = df["total"] / df["principal"] - 1
    df["wealth_multiple"] = df["total"] / df["principal"]
    df["relative_to_principal"] = df["total"] / df["principal"] - 1
    df["drawdown_from_peak"] = df["total"] / df["total"].cummax() - 1
    df["cagr_to_date"] = 0.0

    positive_time = df["year"] > 0
    df.loc[positive_time, "cagr_to_date"] = (
        (df.loc[positive_time, "total"] / principal)
        ** (1 / df.loc[positive_time, "year"])
        - 1
    )

    return df


# ============================================================
# Build schedules
# ============================================================

n_periods = YEARS * COMPOUNDS_PER_YEAR
periods = np.arange(0, n_periods + 1)
time = periods / COMPOUNDS_PER_YEAR

terminal_value = terminal_value_from_cagr(
    initial_value=INITIAL_PRINCIPAL,
    cagr=SPX_CAGR,
    years=YEARS,
)

pre_shock_value = terminal_value_from_cagr(
    initial_value=INITIAL_PRINCIPAL,
    cagr=SPX_CAGR,
    years=DRAWDOWN_YEAR,
)

unhedged_shock_value = pre_shock_value * (1 - UNHEDGED_DRAWDOWN)
hedged_shock_value = pre_shock_value * (1 - HEDGED_DRAWDOWN)

unhedged_vs_principal = unhedged_shock_value / INITIAL_PRINCIPAL - 1
hedged_vs_principal = hedged_shock_value / INITIAL_PRINCIPAL - 1

unhedged_recovery_cagr = (
    (terminal_value / unhedged_shock_value) ** (1 / (YEARS - DRAWDOWN_YEAR)) - 1
)

hedged_recovery_cagr = (
    (terminal_value / hedged_shock_value) ** (1 / (YEARS - DRAWDOWN_YEAR)) - 1
)

# Left panel: stochastic SPX-like Brownian bridge pinned to the 7.5% CAGR endpoint.
left_spx_path = build_log_brownian_bridge_segment(
    start_year=0,
    end_year=YEARS,
    start_value=INITIAL_PRINCIPAL,
    end_value=terminal_value,
    volatility=LEFT_SPX_VOLATILITY,
    seed=LEFT_SPX_SEED,
)

left_spx_path = add_path_metrics(left_spx_path, "Stochastic SPX 7.5% CAGR")

# Right panel: shared stochastic pre-shock path, then two different drawdown/recovery paths.
shared_pre_shock_path = build_log_brownian_bridge_segment(
    start_year=0,
    end_year=DRAWDOWN_YEAR,
    start_value=INITIAL_PRINCIPAL,
    end_value=pre_shock_value,
    volatility=PRE_SHOCK_VOLATILITY,
    seed=PRE_SHOCK_SEED,
)

# Drop the year-5 endpoint from the pre-shock segment and then add two rows at year 5:
# one pre-shock point and one post-shock point. This creates a visible vertical shock.
pre_without_endpoint = shared_pre_shock_path[
    shared_pre_shock_path["year"] < DRAWDOWN_YEAR
].copy()

unhedged_recovery_segment = build_log_brownian_bridge_segment(
    start_year=DRAWDOWN_YEAR,
    end_year=YEARS,
    start_value=unhedged_shock_value,
    end_value=terminal_value,
    volatility=UNHEDGED_RECOVERY_VOLATILITY,
    seed=UNHEDGED_RECOVERY_SEED,
)

hedged_recovery_segment = build_log_brownian_bridge_segment(
    start_year=DRAWDOWN_YEAR,
    end_year=YEARS,
    start_value=hedged_shock_value,
    end_value=terminal_value,
    volatility=HEDGED_RECOVERY_VOLATILITY,
    seed=HEDGED_RECOVERY_SEED,
)

unhedged_path_raw = pd.concat([
    pre_without_endpoint,
    pd.DataFrame({
        "period": [DRAWDOWN_YEAR * COMPOUNDS_PER_YEAR],
        "year": [DRAWDOWN_YEAR],
        "total": [pre_shock_value],
    }),
    pd.DataFrame({
        "period": [DRAWDOWN_YEAR * COMPOUNDS_PER_YEAR],
        "year": [DRAWDOWN_YEAR],
        "total": [unhedged_shock_value],
    }),
    unhedged_recovery_segment[unhedged_recovery_segment["year"] > DRAWDOWN_YEAR],
], ignore_index=True)

hedged_path_raw = pd.concat([
    pre_without_endpoint,
    pd.DataFrame({
        "period": [DRAWDOWN_YEAR * COMPOUNDS_PER_YEAR],
        "year": [DRAWDOWN_YEAR],
        "total": [pre_shock_value],
    }),
    pd.DataFrame({
        "period": [DRAWDOWN_YEAR * COMPOUNDS_PER_YEAR],
        "year": [DRAWDOWN_YEAR],
        "total": [hedged_shock_value],
    }),
    hedged_recovery_segment[hedged_recovery_segment["year"] > DRAWDOWN_YEAR],
], ignore_index=True)

# ------ FORCE FINAL DRAWDOWN AT YEAR 30 FOR BOTH RIGHT-PANEL PATHS ------
# For unhedged, apply 40% drawdown at year 30 (enforce 40% drop from value at year 30)
unhedged_final_idx = (unhedged_path_raw["year"] == YEARS)
if unhedged_final_idx.any():
    cur_val = unhedged_path_raw.loc[unhedged_final_idx, "total"].values[0]
    unhedged_path_raw.loc[unhedged_final_idx, "total"] = cur_val * (1 - FINAL_UNHEDGED_DRAWDOWN)

# For hedged, apply 15% drawdown at year 30 (enforce 15% drop from value at year 30)
hedged_final_idx = (hedged_path_raw["year"] == YEARS)
if hedged_final_idx.any():
    cur_val = hedged_path_raw.loc[hedged_final_idx, "total"].values[0]
    hedged_path_raw.loc[hedged_final_idx, "total"] = cur_val * (1 - FINAL_HEDGED_DRAWDOWN)

unhedged_path = add_path_metrics(
    unhedged_path_raw,
    "Unhedged -40% Drawdown (Final DD at Year 30)"
)
hedged_path = add_path_metrics(
    hedged_path_raw,
    "Crisis Alpha Hedge -15% Drawdown (Final DD at Year 30)"
)

# Smooth reference path only for comparison in hover and optional dotted guide.
reference = pd.DataFrame({
    "period": periods,
    "year": time,
    "principal": INITIAL_PRINCIPAL,
})

reference["total"] = INITIAL_PRINCIPAL * (1 + SPX_CAGR) ** reference["year"]
reference["cumulative_gain"] = reference["total"] - reference["principal"]
reference["cumulative_return"] = reference["total"] / reference["principal"] - 1
reference["wealth_multiple"] = reference["total"] / reference["principal"]

# Align reference values onto right paths for hover comparisons.
for path_df in [unhedged_path, hedged_path]:
    path_df["smooth_reference_value"] = INITIAL_PRINCIPAL * (1 + SPX_CAGR) ** path_df["year"]
    path_df["gap_vs_smooth_reference"] = path_df["total"] / path_df["smooth_reference_value"] - 1

# Summary metrics
left_final_value = left_spx_path["total"].iloc[-1]
left_full_period_cagr = (left_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
left_max_drawdown = left_spx_path["drawdown_from_peak"].min()

unhedged_final_value = unhedged_path["total"].iloc[-1]
hedged_final_value = hedged_path["total"].iloc[-1]

unhedged_full_period_cagr = (unhedged_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1
hedged_full_period_cagr = (hedged_final_value / INITIAL_PRINCIPAL) ** (1 / YEARS) - 1

unhedged_max_drawdown = unhedged_path["drawdown_from_peak"].min()
hedged_max_drawdown = hedged_path["drawdown_from_peak"].min()

# ============================================================
# Styling — carried forward from the uploaded dark Plotly style
# ============================================================

off_white = "#e0e0e0"
principal_color = "#00ff88"
spx_color = "#00d4ff"
unhedged_color = "#f25b37"
hedged_color = "#ffaa33"
reference_color = "#aaaaaa"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

all_visible_totals = pd.concat([
    left_spx_path["total"],
    unhedged_path["total"],
    hedged_path["total"],
    reference["total"],
])

y_max = all_visible_totals.max() * 1.15

# ============================================================
# Subplot-specific metric strings
# ============================================================

left_subtitle = (
    "<b>Stochastic SPX Path · 7.50% Terminal CAGR</b><br>"
    f"Final: ${left_final_value:,.0f} · "
    f"CAGR: {left_full_period_cagr:.2%} · "
    f"Max DD: {left_max_drawdown:.1%}"
)

right_subtitle = (
    "<b>Year 5 Market Drawdown · Final Year Forced Drawdown</b><br>"
    f"Unhedged: -40% at Year 5, -40% at Year 30 · "
    f"Hedged: -15% at Year 5, -15% at Year 30"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(left_subtitle, right_subtitle)
)

# -------------------------
# Left: stochastic SPX CAGR path
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[left_spx_path["year"].iloc[0]],
        y=[left_spx_path["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=True,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[left_spx_path["year"].iloc[0]],
        y=[left_spx_path["total"].iloc[0]],
        customdata=np.column_stack([
            left_spx_path["cumulative_gain"].iloc[:1],
            left_spx_path["cumulative_return"].iloc[:1],
            left_spx_path["wealth_multiple"].iloc[:1],
            left_spx_path["drawdown_from_peak"].iloc[:1],
            left_spx_path["cagr_to_date"].iloc[:1],
        ]),
        mode="lines",
        line=dict(color=spx_color, width=4),
        name="Stochastic SPX Path",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "SPX-style value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "Wealth multiple: %{customdata[2]:.2f}x<br>"
            "Drawdown from peak: %{customdata[3]:.2%}<br>"
            "CAGR to date: %{customdata[4]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[left_spx_path["year"].iloc[0]],
        y=[left_spx_path["total"].iloc[0]],
        customdata=np.column_stack([
            left_spx_path["wealth_multiple"].iloc[:1],
            left_spx_path["cumulative_return"].iloc[:1],
            left_spx_path["drawdown_from_peak"].iloc[:1],
        ]),
        mode="markers",
        marker=dict(color=spx_color, size=10, line=dict(color=off_white, width=1)),
        name="SPX Current Value",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Current value: %{y:$,.0f}<br>"
            "Wealth multiple: %{customdata[0]:.2f}x<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "Drawdown from peak: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# -------------------------
# Right: stochastic drawdown paths, no text overlays
# -------------------------

fig.add_trace(
    go.Scatter(
        x=[reference["year"].iloc[0]],
        y=[reference["principal"].iloc[0]],
        mode="lines",
        line=dict(color=principal_color, width=3, dash="dash"),
        opacity=0.75,
        name="Principal Baseline",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Principal: %{y:$,.0f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[reference["year"].iloc[0]],
        y=[reference["total"].iloc[0]],
        customdata=np.column_stack([
            reference["cumulative_gain"].iloc[:1],
            reference["cumulative_return"].iloc[:1],
        ]),
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dot"),
        opacity=0.55,
        name="Smooth 7.5% Guide",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Smooth 7.5% value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[unhedged_path["year"].iloc[0]],
        y=[unhedged_path["total"].iloc[0]],
        customdata=np.column_stack([
            unhedged_path["cumulative_gain"].iloc[:1],
            unhedged_path["cumulative_return"].iloc[:1],
            unhedged_path["relative_to_principal"].iloc[:1],
            unhedged_path["gap_vs_smooth_reference"].iloc[:1],
            unhedged_path["drawdown_from_peak"].iloc[:1],
            unhedged_path["cagr_to_date"].iloc[:1],
        ]),
        mode="lines",
        line=dict(color=unhedged_color, width=4),
        name="Unhedged Path",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Unhedged value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "Relative to principal: %{customdata[2]:+.2%}<br>"
            "Gap vs smooth guide: %{customdata[3]:+.2%}<br>"
            "Drawdown from peak: %{customdata[4]:.2%}<br>"
            "CAGR to date: %{customdata[5]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[hedged_path["year"].iloc[0]],
        y=[hedged_path["total"].iloc[0]],
        customdata=np.column_stack([
            hedged_path["cumulative_gain"].iloc[:1],
            hedged_path["cumulative_return"].iloc[:1],
            hedged_path["relative_to_principal"].iloc[:1],
            hedged_path["gap_vs_smooth_reference"].iloc[:1],
            hedged_path["drawdown_from_peak"].iloc[:1],
            hedged_path["cagr_to_date"].iloc[:1],
        ]),
        mode="lines",
        line=dict(color=hedged_color, width=4),
        name="Crisis Alpha Hedge",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Hedged value: %{y:$,.0f}<br>"
            "Cumulative gain: %{customdata[0]:$,.0f}<br>"
            "Cumulative return: %{customdata[1]:.2%}<br>"
            "Relative to principal: %{customdata[2]:+.2%}<br>"
            "Gap vs smooth guide: %{customdata[3]:+.2%}<br>"
            "Drawdown from peak: %{customdata[4]:.2%}<br>"
            "CAGR to date: %{customdata[5]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[unhedged_path["year"].iloc[0]],
        y=[unhedged_path["total"].iloc[0]],
        customdata=np.column_stack([
            unhedged_path["wealth_multiple"].iloc[:1],
            unhedged_path["relative_to_principal"].iloc[:1],
            unhedged_path["drawdown_from_peak"].iloc[:1],
        ]),
        mode="markers",
        marker=dict(color=unhedged_color, size=10, line=dict(color=off_white, width=1)),
        name="Unhedged Current Value",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Unhedged current value: %{y:$,.0f}<br>"
            "Wealth multiple: %{customdata[0]:.2f}x<br>"
            "Relative to principal: %{customdata[1]:+.2%}<br>"
            "Drawdown from peak: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[hedged_path["year"].iloc[0]],
        y=[hedged_path["total"].iloc[0]],
        customdata=np.column_stack([
            hedged_path["wealth_multiple"].iloc[:1],
            hedged_path["relative_to_principal"].iloc[:1],
            hedged_path["drawdown_from_peak"].iloc[:1],
        ]),
        mode="markers",
        marker=dict(color=hedged_color, size=10, line=dict(color=off_white, width=1)),
        name="Hedged Current Value",
        showlegend=False,
        hovertemplate=(
            "Year: %{x:.0f}<br>"
            "Hedged current value: %{y:$,.0f}<br>"
            "Wealth multiple: %{customdata[0]:.2f}x<br>"
            "Relative to principal: %{customdata[1]:+.2%}<br>"
            "Drawdown from peak: %{customdata[2]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Horizontal principal reference line on both charts
for col in [1, 2]:
    fig.add_hline(
        y=INITIAL_PRINCIPAL,
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        row=1,
        col=col
    )

# Year-5 shock guide only; no annotation text overlays.
fig.add_vline(
    x=DRAWDOWN_YEAR,
    line=dict(color=unhedged_color, width=1.5, dash="dash"),
    opacity=0.55,
    row=1,
    col=2
)

# Year-30 forced drawdown guide; no annotation (NEW)
fig.add_vline(
    x=YEARS,
    line=dict(color="#feae09", width=1.5, dash="dot"),
    opacity=0.45,
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

# Trace index map:
# 0: left principal baseline
# 1: left stochastic SPX path
# 2: left current marker
# 3: right principal baseline
# 4: right smooth guide
# 5: right unhedged stochastic path
# 6: right hedged stochastic path
# 7: right unhedged current marker
# 8: right hedged current marker

for yr in range(0, YEARS + 1):
    frame_name = f"year_{yr}"

    left_slice = left_spx_path[left_spx_path["year"] <= yr]
    reference_slice = reference[reference["year"] <= yr]
    unhedged_slice = unhedged_path[unhedged_path["year"] <= yr]
    hedged_slice = hedged_path[hedged_path["year"] <= yr]

    # protect for empty slice at final step
    if not left_slice.empty:
        left_current = left_spx_path[left_spx_path["year"] == yr].iloc[-1]
    else:
        left_current = left_spx_path.iloc[0]
    if not unhedged_slice.empty:
        unhedged_current = unhedged_path[unhedged_path["year"] == yr].iloc[-1]
    else:
        unhedged_current = unhedged_path.iloc[0]
    if not hedged_slice.empty:
        hedged_current = hedged_path[hedged_path["year"] == yr].iloc[-1]
    else:
        hedged_current = hedged_path.iloc[0]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=left_slice["year"],
                    y=left_slice["principal"]
                ),
                go.Scatter(
                    x=left_slice["year"],
                    y=left_slice["total"],
                    customdata=np.column_stack([
                        left_slice["cumulative_gain"],
                        left_slice["cumulative_return"],
                        left_slice["wealth_multiple"],
                        left_slice["drawdown_from_peak"],
                        left_slice["cagr_to_date"],
                    ])
                ),
                go.Scatter(
                    x=[left_current["year"]],
                    y=[left_current["total"]],
                    customdata=np.array([[
                        left_current["wealth_multiple"],
                        left_current["cumulative_return"],
                        left_current["drawdown_from_peak"],
                    ]])
                ),
                go.Scatter(
                    x=reference_slice["year"],
                    y=reference_slice["principal"]
                ),
                go.Scatter(
                    x=reference_slice["year"],
                    y=reference_slice["total"],
                    customdata=np.column_stack([
                        reference_slice["cumulative_gain"],
                        reference_slice["cumulative_return"],
                    ])
                ),
                go.Scatter(
                    x=unhedged_slice["year"],
                    y=unhedged_slice["total"],
                    customdata=np.column_stack([
                        unhedged_slice["cumulative_gain"],
                        unhedged_slice["cumulative_return"],
                        unhedged_slice["relative_to_principal"],
                        unhedged_slice["gap_vs_smooth_reference"],
                        unhedged_slice["drawdown_from_peak"],
                        unhedged_slice["cagr_to_date"],
                    ])
                ),
                go.Scatter(
                    x=hedged_slice["year"],
                    y=hedged_slice["total"],
                    customdata=np.column_stack([
                        hedged_slice["cumulative_gain"],
                        hedged_slice["cumulative_return"],
                        hedged_slice["relative_to_principal"],
                        hedged_slice["gap_vs_smooth_reference"],
                        hedged_slice["drawdown_from_peak"],
                        hedged_slice["cagr_to_date"],
                    ])
                ),
                go.Scatter(
                    x=[unhedged_current["year"]],
                    y=[unhedged_current["total"]],
                    customdata=np.array([[
                        unhedged_current["wealth_multiple"],
                        unhedged_current["relative_to_principal"],
                        unhedged_current["drawdown_from_peak"],
                    ]])
                ),
                go.Scatter(
                    x=[hedged_current["year"]],
                    y=[hedged_current["total"]],
                    customdata=np.array([[
                        hedged_current["wealth_multiple"],
                        hedged_current["relative_to_principal"],
                        hedged_current["drawdown_from_peak"],
                    ]])
                ),
            ],
            traces=list(range(9)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": True},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": str(yr),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    # No main title — matching the corrected dark Plotly style
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=85, b=170, r=50, l=75),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.16,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": True},
                        "transition": {"duration": 80},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": True},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 80},
        "pad": {"b": 40, "t": 30},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }],
    annotations=list(fig.layout.annotations)
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=2
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, y_max],
    title_text="Stochastic 7.5% CAGR Path",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[-0.35, YEARS + 0.35],
    title_text="Year",
    dtick=2
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, y_max],
    title_text="Same Terminal CAGR, Forced Final Drawdown",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================
fig.show()

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
 - This notebook explains how portfolio wealth grows through the geometric compounding of returns, and why compound (geometric) growth diverges dramatically from simple (arithmetic) interest over long horizons.
 - It demonstrates that very different year-to-year return paths can compound to nearly identical terminal values, so over long horizons the specific path matters far less than the effective rate you compound at.
 - Compound Annual Growth Rate (CAGR) is derived as $r = (V_T / V_0)^{1/T} - 1$, summarizing multi-year performance as a single steady annual rate—as if growth had been constant each year.
 - Practical rules of thumb are covered: the Rule of 72 (72 ÷ CAGR ≈ years for principal to double) and a "BS test" reminder that outrageous returns almost always imply outrageous risk—there is no free lunch.
 - The key message: CAGR is a tool for describing and comparing historical growth across different states of the world, **not** for predicting the future—historical performance is not indicative of what's to come.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$